# 使用 Tidyverse 进行数据整理与操作

跨语言数据管道：Python 下载数据 → R 使用 dplyr 处理 → Python 进行结果可视化。

展示 **SharedVFS** — 允许 Python 和 R 互相交换文件的主机共享文件系统。

## 1. Python：下载数据集

In [ ]:
import micropip
await micropip.install('pandas')
import pandas as pd, pyodide.http, os

url = "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv"
resp = await pyodide.http.pyfetch(url)
text = await resp.string()

os.makedirs("/shared/data", exist_ok=True)
with open("/shared/data/gapminder.csv", "w") as f:
    f.write(text)

df = pd.read_csv("/shared/data/gapminder.csv")
print(f"Downloaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. R：安装 dplyr + tidyr 并读取共享数据

In [ ]:
install.packages(c("dplyr", "tidyr"))
library(dplyr)

gap <- read.csv("/shared/data/gapminder.csv")
cat("Read from SharedVFS:", nrow(gap), "rows\n")
glimpse(gap)

## 3. dplyr：按大洲汇总（2007 年）

In [ ]:
gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    countries = n(),
    mean_life = round(mean(lifeExp), 1),
    median_gdp = round(median(gdpPercap), 0),
    total_pop = sum(as.numeric(pop))
  ) %>%
  arrange(desc(mean_life))

## 4. dplyr：预期寿命增长最多的国家

In [ ]:
gains <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  select(country, continent, year, lifeExp) %>%
  tidyr::pivot_wider(names_from = year, values_from = lifeExp,
                     names_prefix = "y") %>%
  mutate(gain = y2007 - y1952) %>%
  arrange(desc(gain)) %>%
  head(10)
gains

## 5. dplyr：各大洲人口增长情况

In [ ]:
pop_growth <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  group_by(continent, year) %>%
  summarize(total_pop = sum(as.numeric(pop)), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = year, values_from = total_pop,
                     names_prefix = "pop_") %>%
  mutate(growth_pct = round((pop_2007 / pop_1952 - 1) * 100, 1)) %>%
  arrange(desc(growth_pct))
pop_growth

## 6. R：将结果写入 SharedVFS

In [ ]:
# Write continent summary for Python to visualize
summary_2007 <- gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    mean_life = round(mean(lifeExp), 1),
    mean_gdp = round(mean(gdpPercap), 0),
    .groups = "drop"
  )
write.csv(summary_2007, "/shared/data/r_summary.csv", row.names = FALSE)
cat("Wrote /shared/data/r_summary.csv\n")
summary_2007

## 7. Python：可视化 R 的输出结果

In [ ]:
import micropip
await micropip.install('plotly')
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

r_summary = pd.read_csv("/shared/data/r_summary.csv")
print("Read from SharedVFS (written by R):")
print(r_summary.to_string(index=False))

fig = px.bar(r_summary, x="continent", y="mean_life",
             title="Mean Life Expectancy by Continent (2007) — from R → Python",
             labels={"mean_life": "Life Expectancy (years)", "continent": "Continent"},
             color="continent")
fig.update_layout(template='plotly_dark', showlegend=False)
show_plotly(fig)

In [ ]:
fig = px.scatter(r_summary, x="mean_gdp", y="mean_life",
                 text="continent", size=[40]*len(r_summary),
                 title="GDP vs Life Expectancy by Continent (R summary → Python plot)",
                 labels={"mean_gdp": "Mean GDP per Capita", "mean_life": "Mean Life Expectancy"})
fig.update_traces(textposition="top center")
fig.update_layout(template='plotly_dark')
fig.update_yaxes(range=[r_summary['mean_life'].min() - 2, r_summary['mean_life'].max() + 6])
show_plotly(fig)

## 核心要点

- **Python** 将 CSV 数据下载到了 `/shared/data/`
- **R** 通过 SharedVFS 读取了数据，并使用 dplyr 管道进行了数据处理
- **R** 将汇总数据写回至 `/shared/data/r_summary.csv`
- **Python** 读取了 R 的输出并使用 Plotly 创建了交互式图表

所有文件共享均通过 SharedVFS 进行 — 无需手动导入或导出。